In [ ]:
from pathlib import Path
import shutil

src_root = Path("D:\dsp proj\content")

for folder in ["fake_images"]:
    src = src_root / folder
    dst = src_root / f"{folder}_flat"
    dst.mkdir(exist_ok=True)

    for p in src.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".jpg"}:
            target = dst / p.name

            if target.exists():
                stem, suffix = p.stem, p.suffix
                i = 1
                while (dst / f"{stem}_{i}{suffix}").exists():
                    i += 1
                target = dst / f"{stem}_{i}{suffix}"

            shutil.copy2(p, target)

print("Done")

In [1]:
from pathlib import Path
import shutil

src_root = Path(r"D:/dsp proj/content")
folder = "real_images"
src = src_root / folder
dst = src_root / f"{folder}_flat"
dst.mkdir(exist_ok=True)

print("SRC exists:", src.exists(), src)
print("DST exists:", dst.exists(), dst)

copied = 0
for p in src.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        target = dst / p.name
        if target.exists():
            stem, suffix = p.stem, p.suffix
            i = 1
            while (dst / f"{stem}_{i}{suffix}").exists():
                i += 1
            target = dst / f"{stem}_{i}{suffix}"
        shutil.copy2(p, target)
        copied += 1

print("Copied files:", copied)

SRC exists: True D:\dsp proj\content\real_images
DST exists: True D:\dsp proj\content\real_images_flat


KeyboardInterrupt: 

In [ ]:
# hf dataset = ShreyashDhoot/AI_vs_Real

In [3]:
from pathlib import Path
from datasets import Dataset, Image
from PIL import Image as PILImage

# Configuration
content_root = Path(r"D:\dsp proj\content")
real_folder = content_root / "real"
fake_folder = content_root / "ai"

# Collect image data with labels and indices
data = {
    "index": [],
    "image": [],
    "label": [],
    "filename": []
}

idx = 0

# Process real images (label = 1)
if real_folder.exists():
    for img_path in sorted(real_folder.glob("*.jpg")) + sorted(real_folder.glob("*.jpeg")) + sorted(real_folder.glob("*.png")):
        if img_path.is_file():
            try:
                # Verify image can be opened
                PILImage.open(img_path)
                data["index"].append(idx)
                data["image"].append(str(img_path))
                data["label"].append(1)
                data["filename"].append(img_path.name)
                idx += 1
            except Exception as e:
                print(f"Skipping invalid image {img_path.name}: {e}")

# Process fake images (label = 0)
if fake_folder.exists():
    for img_path in sorted(fake_folder.glob("*.jpg")) + sorted(fake_folder.glob("*.jpeg")) + sorted(fake_folder.glob("*.png")):
        if img_path.is_file():
            try:
                # Verify image can be opened
                PILImage.open(img_path)
                data["index"].append(idx)
                data["image"].append(str(img_path))
                data["label"].append(0)
                data["filename"].append(img_path.name)
                idx += 1
            except Exception as e:
                print(f"Skipping invalid image {img_path.name}: {e}")

print(f"Collected {idx} images total")
print(f"Real images (label=1): {sum(1 for l in data['label'] if l == 1)}")
print(f"Fake images (label=0): {sum(1 for l in data['label'] if l == 0)}")


Collected 180477 images total
Real images (label=1): 72386
Fake images (label=0): 108091


In [4]:
# Create Hugging Face dataset
dataset = Dataset.from_dict(data)

# Cast image column to Image type
dataset = dataset.cast_column("image", Image())

print("Dataset structure:")
print(dataset)
print("\nFirst few examples:")
for i in range(min(3, len(dataset))):
    example = dataset[i]
    print(f"Index {example['index']}: {example['filename']} - Label {example['label']}")


Dataset structure:
Dataset({
    features: ['index', 'image', 'label', 'filename'],
    num_rows: 180477
})

First few examples:
Index 0: 1.jpg - Label 1
Index 1: 10.jpg - Label 1
Index 2: 100.jpg - Label 1


In [5]:
from huggingface_hub import login

# Login to Hugging Face (follow the prompt to enter your token)
login()

# Push dataset to Hugging Face hub
repo_id = "ShreyashDhoot/AI_vs_Real"

print(f"Pushing dataset to {repo_id}...")
dataset.push_to_hub(repo_id, private=False)
print("Upload complete!")


Pushing dataset to ShreyashDhoot/AI_vs_Real...


Uploading the dataset shards:   0%|          | 0/5 [00:00<?, ? shards/s]

Map:   0%|          | 0/36096 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/36095 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete!
